# MiniProject #01: Cobblestone Gifts E-Commerce Data

by Patrick Donnelly & Burke Havranek

EECE 6544: Introduction to Machine Learning and Pattern Recognition

Northeastern University College of Engineering

Summer 2026, Session B

## Scenario

You have just joined the analytics team at **Cobblestone Gifts**, a UK-based online retailer that sells all-occasion
novelty gifts to both wholesale buyers and individual shoppers. The company has never run a formal sales review.
With the financial year closing, the Head of Commercial wants a clear picture of what sold, to whom, and where
but the only thing the IT team can hand you is a single raw export pulled straight from the order system: roughly
half a million transaction lines from December 2010 to December 2011.

The export has never been touched. It mixes completed sales with **cancelled orders and returns**, is missing a
customer ID on a large share of rows, contains **blank product descriptions** and **duplicate lines**, and is peppered
with non-product entries such as postage, bank charges, and manual adjustments. Prices and quantities include
impossible values. In short: **no number in this file can be trusted yet**.

Your job is to clean it. You will profile the data, make and document defensible cleaning decisions, build a tidy
*completed-sales* dataset, and then use it to answer a set of real business questions. The Head of Commercial
will read your findings so every claim must rest on data you can stand behind.

## Part 1: Clean the Data

### Phase A: Load & Profile

#### #3.2: Getting Information

In [ ]:
#!/bin/python3
# --Beginning of Code--
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "pandas", "kagglehub", "caseutil")

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 80)
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
import caseutil

path = kagglehub.dataset_download("carrie1/ecommerce-data")
csv_file = os.path.join(path, "data.csv")
df = pd.read_csv(csv_file, encoding='ISO-8859-1')

In [ ]:
print(df.shape) # Note the total rows
N_ROW, _ = df.shape

In [ ]:
df.head()

In [ ]:
df.info() # Note the populated rows

In [ ]:
df.describe()

From the above, we find 541909 total entries in the database, of which six field (InvoiceNo, StockCode, Quantity, InvoiceDate, UnitPrice, Country) are fully-populated with non-null data. Of the remaining two columns, `Description` is missing 1454 entries (~0.268%), whereas `CustomerID` is missing 135080 entries (~24.9%).

#### #3.9 Unique Values

In [ ]:
# Naive count of all unique elements in each column
df[["Country", "StockCode", "Description"]].nunique()

In [ ]:
# Enumeration of each column with and without consideration of null columns
df["Country"].value_counts()

In [ ]:
df["Country"].value_counts(dropna=False)

In [ ]:
df["StockCode"].value_counts()

In [ ]:
df["StockCode"].value_counts(dropna=False)

In [ ]:
df["Description"].value_counts()

In [ ]:
df["Description"].value_counts(dropna=False)

In [ ]:
# Isolate non product codes
NON_PRODUCT = ["POST", "DOT", "M", "BANK CHARGES", "AMAZONFEE", "D"]
non_product = df[df["StockCode"].isin(NON_PRODUCT)]


In [ ]:
non_product["StockCode"].value_counts()

#### #3.8: min/max/sum/mean/count

In [ ]:
# Summarize quantity and unit price data (including returns)
df[["Quantity", "UnitPrice"]].describe()

In [ ]:
# Identify returns
returns = df[df["Quantity"] < 0]
returns

In [ ]:
# Identify invalid prices
invalid_prices = df[df["UnitPrice"] < 0]
invalid_prices

#### # 3.1 Creating a DataFrame

In [ ]:
# Dictionary for right now to consolidate some regions
# Follows format following later replacement of EIRE, RSA, USA, Unspecified
COUNTRY_TO_REGION = {
    # UK/Ireland
    "United Kingdom": "UK&IE",
    "Ireland": "UK&IE",
    "Channel Islands": "UK&IE",

    # Mainland Europe (core)
    "Germany": "Continental Europe",
    "France": "Continental Europe",
    "Spain": "Continental Europe",
    "Netherlands": "Continental Europe",
    "Belgium": "Continental Europe",
    "Switzerland": "Continental Europe",
    "Portugal": "Continental Europe",
    "Italy": "Continental Europe",
    "Austria": "Continental Europe",
    "Denmark": "Continental Europe",
    "Norway": "Continental Europe",
    "Sweden": "Continental Europe",
    "Finland": "Continental Europe",
    "Iceland": "Continental Europe",
    "Cyprus": "Continental Europe",
    "Greece": "Continental Europe",
    "Malta": "Continental Europe",
    "Lithuania": "Continental Europe",
    "Poland": "Continental Europe",
    "Czech Republic": "Continental Europe",
    "European Community": "Continental Europe",

    # Middle East
    "Israel": "Middle East",
    "Lebanon": "Middle East",
    "United Arab Emirates": "Middle East",
    "Bahrain": "Middle East",
    "Saudi Arabia": "Middle East",

    # Africa
    "South Africa": "Africa",

    # Americas
    "United States": "Americas",
    "Canada": "Americas",
    "Brazil": "Americas",

    # Oceania
    "Australia": "Oceania",
    "Japan": "Oceania",
    "Hong Kong": "Oceania",
    "Singapore": "Oceania",

    # Unknown
    "": ""
}

### Phase B: Select & filter

#### #3.3: Slicing

In [ ]:
df.iloc[2]

In [ ]:
num_entries = df.shape[0]
Incramented_Index = list(range(0,(num_entries)))
df["Unique_Index"]= Incramented_Index
df.set_index('Unique_Index', inplace=True)
df.loc[2]

#### #3.4: Conditional Selection

In [ ]:
canceled_order_group = df['InvoiceNo'].astype(str).str.startswith('c')
print(canceled_order_group)


In [ ]:
zero_quantity_group = df['Quantity'] <= 0
print(zero_quantity_group)

In [ ]:
zero_unitprice_group = df['UnitPrice'] <= 0
print(zero_quantity_group)

#### #3.5: Sorting Values

In [ ]:
df = df.sort_values(by="Quantity", ascending=False)

In [ ]:
df['return'] = df['Quantity']*df['UnitPrice']
df =df.sort_values('return', ascending=False)
df = df.drop(columns='return')
df


### Phase C: Clean & Fix

#### #3.6: Replacing Values

In [ ]:
# Lengthen initialisms, make Unspecified nations null
MAP = {"EIRE": "Ireland", "RSA": "South Africa", "USA": "United States", "Unspecified": ""}
df["Country"] = df["Country"].map(lambda x: MAP[x] if x in MAP else x)
df["Country"].value_counts()

#### #3.7: Renaming Columns

In [ ]:
# Use standard library to convert each column to snake_case
MAP = {x: caseutil.to_snake(x) for x in df.columns}
df.rename(columns=MAP, inplace=True)
df

#### #3.10: Handling Missing Values

In [ ]:
df[df["description"].isnull()]

In [ ]:
# Eliminate items lacking a description
df["description"].dropna(inplace=True)
df = df[~df["description"].isnull()]

In [ ]:
df[df["customer_id"].isnull()]

In [ ]:
# Keep unknown CustomerID numbers
df["customer_id"].fillna(0, inplace=True)
df

Whereas relatively few entries (~0.268%) lack a description, nearly a quarter (~24.9%) lack a customer ID. Given the marginal number of entries lacking a description, the significance of product descriptions in noting consumer trends, and the fact that a lack of description appears to correlate with missing customer IDs and unit prices, these data should be excluded, and are able to be done so with minimal effect on the rest of the set. 

By contrast, the number of missing customer IDs is much too significant to exclude, and do not appear to correlate with any other missing data. Despite the fact that these entries are lacking information regarding specific customers, they still provide valuable data regarding consumer trends by both country and description.

#### #3.11: Deleting a Column

In [ ]:
# Drop stock_code, as it is shadowed by description
df = df.drop(columns="stock_code")
df

The `stock_code` column is redundant, carrying similar information to `description` and `quantity`, both of which are more useful columns to sort the data. Descriptions give a meaningful explanation of what was sold, or what transpired, and the quantity magnitude indicates returns, which renders the stock code redundant.

#### #3.12: Deleting a Row

In [ ]:
# Delete cancellations, non-product lines, and impossible prices/quantities
df = df[df["quantity"] > 0] # Remove cancellations
df = df[~df["description"].isin(NON_PRODUCT)] # Remove non-products
df = df[df["unit_price"] > 0] # Remove impossible prices/quantities
df

#### #3.13 Dropping Duplicates

In [ ]:
print(f"Duplicate rows detected: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
df

### Phase D: Engineer & Summarize

#### #3.18: Applying a Function

In [ ]:
df['revenue'] = df['quantity']*df['unit_price'] # creates a revenue row that gets used in the future
df

In [ ]:
df ['description'].apply(str.strip) # removes leading and tailing spaces
df ['description'].apply(str.title) # sets all text to title case to standardize
df

#### #3.17: Looping over a Column

In [ ]:
for index, row in df.iterrows():
    print(row['description'])

This loop takes signifigently longer to perform then the applies do. It also forces you to go through all elements of a row where apply can just focus on the elements you care about.

#### #3.14: Grouping by Values

In [ ]:
df.groupby('customer_id').sum('revenue')

In [ ]:
df.groupby('country').sum('revenue')

#### #3.16: Aggregating

In [ ]:
df.groupby('country').agg({'revenue':['sum','mean','count']})

In [ ]:
df.groupby('customer_id').agg({'revenue':['sum','mean','count']})

#### #3.19: Applying to Groups

In [ ]:
def custom_summary_report(group): # functions are easier to use in apply
    total_rev = (group['revenue']).sum()

    months = group['invoice_date'].dt.to_period('M').nunique()
    
    return pd.Series({
        'total_revenue': total_rev,
        'active_months': months
    })

df["invoice_date"] = pd.to_datetime(df["invoice_date"], format="%m/%d/%Y %H:%M") # Sets the invoice_date to a consistant time structure for better use in the future
df.groupby('customer_id').apply(custom_summary_report)



#### #3.15: Grouping by Time

In [ ]:
df = df.set_index('invoice_date')

In [ ]:
monthly_revenue = df['revenue'].resample('ME').sum()
monthly_revenue.plot(title='Monthly Revenue Trend')

In [ ]:
weekly_revenue = df['revenue'].resample('W').sum()
weekly_revenue.plot(title='Monthly Revenue Trend')

### Phase E: Combine

#### #3.20: Concatenating

In [ ]:
# Change date format into ISO sortable format
# df["invoice_date"] = pd.to_datetime(df["invoice_date"], format="%m/%d/%Y %H:%M") # was moved further up for part D
df = df.sort_index(ascending=True)
df

In [ ]:
# Take rough midpoint of year and split table
split_date = '2011-06-15 00:00:00' # June 15th is the approximate midpoint of data range

# df1 = df[df["invoice_date"] < split_date] # this code was made before invoice was set as index
df1 = df.loc[df.index < split_date]
df1


In [ ]:
# df2 = df[df["invoice_date"] > split_date] # this code was made before the invoice was set as index
df2 = df.loc[df.index >= split_date]
df2

In [ ]:
# Recombine
df_cat = pd.concat([df1, df2])
df_cat = df_cat.sort_index(ascending=True)
df_cat

In [ ]:
# Verify recombination
assert df.equals(df_cat)

#### #3.21: Merging

In [ ]:
df_regions = pd.DataFrame({"country": list(COUNTRY_TO_REGION.keys()), "region": list(COUNTRY_TO_REGION.values())})
df_left = df.merge(df_regions, how="left")
df_left

In [ ]:
df_inner = df.merge(df_regions, how="inner")
df_inner

In [ ]:
# Verify
assert df_left.equals(df_inner)
df = df_inner

In this particular instance, because all countries have been given their own mapping, both `df_left` and `df_inner` are identical. That is, the method of merge does not matter. Where not all countries are given a mapping, `how='left'` would maintain rows in the original database, converting all unmatched items to `NaN`, whereas `how='inner'` would merge rows only where the two overlap (that is, only where each country has an associated key).

## Part 2: Business Questions

### 1. Seasonality

### 2. Best Sellers

### 3. Markets

### 4. Customer Concentration

### 5. Order Value

### 6. Returns & Cancellations

### 7. Data-Quality Memo